# Subsample JEPA

Hypothesis: the T-Net should map **different numerical observations of the same
function** to similar latents.

```text
same equation f
  ├── view 0 -> T-Net -> z_0 -> symbolic decoder -> CE loss
  └── view 1 -> T-Net -> z_1
                 subsample_loss = mean over pairs of (1 - cos(z_i, z_j))
                 total = CE + lambda * subsample_loss
```

No symbolic-expression target, no MLP predictor, no `[PRED]` tokens.
Separate checkpoints from `jepa_sweep`.

In [ ]:
# Environment setup — works on both Colab and local
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/drive/MyDrive/Symba/symbolic-jepa'
    if not os.path.exists(REPO_DIR):
        %cd /content/drive/MyDrive/Symba
        !git clone https://github.com/zzpDavid2/symbolic-jepa.git {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

    !pip install -q sympy scipy

    CKPT_DIR = '/content/drive/MyDrive/Symba/symbolic-jepa/checkpoints/subsample_jepa'
    LOG_DIR  = '/content/drive/MyDrive/Symba/symbolic-jepa/runs_subsample'
else:
    CKPT_DIR = 'checkpoints/subsample_jepa'
    LOG_DIR  = 'runs_subsample'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')
print(f'Checkpoints: {CKPT_DIR}')

In [ ]:
if IN_COLAB:
    %cd /content/drive/MyDrive/Symba/symbolic-jepa
    !git pull

In [ ]:
import gc
import json
import random
import time

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from symbolic_jepa import (
    PrefixTokenizer, Expression,
    TNet, SymbolicTransformer,
    subsample_consistency_loss,
    PointCloudDataset, MultiViewPointCloudDataset,
    build_multiview_synthetic_splits, load_synthetic_pkl,
    teacher_forced_counts, evaluate_predictions,
    view_consistency_diagnostics, generate_diagnostic_embeddings,
)
from symbolic_jepa.tokenizer import prefix_to_sympy

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ── Model / training hyperparameters (unchanged from jepa_sweep) ──
MAX_VARS    = 1               # univariate synthetic data
D_INPUT     = MAX_VARS + 1    # (x, y) = 2
N_POINTS    = 1000
MAX_SEQ     = 64
D_MODEL     = 512
N_HEADS     = 8
N_LAYERS    = 4
DROPOUT     = 0.2

EPOCHS      = 30
LR          = 3e-4
BATCH       = 16
VAL_EVERY   = 1
USE_AMP     = True

# DataLoader workers: 0.
#
# Under fork (Colab/Linux), workers are forked one at a time, so with
# num_workers >= 2 the SECOND worker inherits a heap copy of its own loader's
# iterator -- which by then already holds worker 0's Process object, tagged
# with the PARENT's pid. If that inherited object is ever finalised inside the
# child, Process.is_alive() trips:
#     AssertionError: can only test a child process
# It is a single loader inheriting from itself; it needs no second loader and
# no persistent_workers to happen, which is why restructuring the loaders did
# not stop it. Measured cost of 0 workers here: ~4 s/epoch (~10% of an epoch).
#
# num_workers = 1 is also safe for this mechanism (that worker inherits an
# EMPTY _workers list, and the pin-memory thread is created after the fork),
# and keeps the prefetch overlap. Raise to 1 if you want that ~10% back.
NUM_WORKERS = 0

# ── Subsample-JEPA config ──
N_VIEWS          = 2       # views per equation during training
TRAIN_VIEW_SEED  = 1729    # governs training views (independent of model seed)
EVAL_SEED        = 2718    # governs fixed diagnostic views
DIAG_N_VIEWS     = 10      # views per equation for the consistency diagnostic
DIAG_N_EXPRS     = 50      # val equations used for the diagnostic

# Which cosine the LOSS optimises (the diagnostics always report both).
#   'centered' — subtract each view's batch mean first.  RECOMMENDED.
#   'cosine'   — ordinary pairwise cosine on the raw embeddings.
# The T-Net max-pools ReLU features into the positive orthant, so raw cosine
# between any two embeddings sits near 0.99 and the raw loss starts ~5e-4:
# too small to produce gradient. In a d_model=128 A/B the raw objective at
# lambda=0.3 left val loss identical to baseline to 4 decimals and moved
# same-function cosine by -0.002 (noise), while centered moved it +0.031 and
# cut its own objective 4.5x. Raw is also *minimised* by collapse (all-equal
# embeddings drive it to 0); centered *penalises* collapse (drives it to 1).
SUBSAMPLE_LOSS = 'centered'

# ── Sweep config ──
LAMBDA_VALUES = [0, 0.003, 0.01, 0.03, 0.1, 0.3]
SEEDS         = [42, 123, 7]
VERSION_TAG   = f'subsample_{SUBSAMPLE_LOSS}_v1'

# Synthetic data (pre-generated by SYMBA_Reg_Data_Gen notebook)
SYNTH_PKL   = 'data/synthetic.pkl'
MAX_SYNTH   = 10_000
SYNTH_SEED  = 42

## Load synthetic data

In [ ]:
tokenizer = PrefixTokenizer(max_vars=MAX_VARS)
print(f'Vocab size: {len(tokenizer)}')

print(f'Loading synthetic expressions from {SYNTH_PKL}...')
synth_exprs = load_synthetic_pkl(
    SYNTH_PKL, max_seq_len=MAX_SEQ,
    tokenizer=tokenizer, max_expressions=MAX_SYNTH,
)
print(f'Loaded {len(synth_exprs)} expressions')

In [ ]:
# Train is multi-view; val/test stay deterministic single-view.
# Same shuffle/seed as build_synthetic_splits, so the partition matches
# the jepa_sweep runs and results are comparable.
synth_train, synth_val, synth_test = build_multiview_synthetic_splits(
    synth_exprs, tokenizer,
    n_points=N_POINTS, max_seq_len=MAX_SEQ, max_vars=MAX_VARS,
    seed=SYNTH_SEED, n_views=N_VIEWS, train_view_seed=TRAIN_VIEW_SEED,
)

## Training / evaluation functions

In [ ]:
from torch.utils.tensorboard import SummaryWriter

In [ ]:
def seed_everything(seed):
    """Full re-seed for model init and training stochasticity."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed + worker_id)
    random.seed(worker_seed + worker_id)

def build_model(tokenizer, dropout=DROPOUT):
    encoder = TNet(d_input=D_INPUT, d_model=D_MODEL)
    model = SymbolicTransformer(
        encoder=encoder, vocab_size=len(tokenizer),
        d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
        d_ff=4 * D_MODEL, max_seq_len=MAX_SEQ,
        dropout=dropout, pad_id=tokenizer.pad_id,
    ).to(DEVICE)
    return model, encoder

def amp_context():
    if USE_AMP and DEVICE == 'cuda':
        return lambda: torch.autocast('cuda', dtype=torch.bfloat16)
    if USE_AMP and DEVICE == 'mps':
        return lambda: torch.autocast('mps', dtype=torch.float16)
    return lambda: torch.amp.autocast('cpu', enabled=False)

def view_consistency(model, val_ds):
    """same/diff-function cosine on fixed diagnostic views."""
    z = generate_diagnostic_embeddings(
        val_ds, model.encoder,
        n_exprs=DIAG_N_EXPRS, n_views=DIAG_N_VIEWS,
        eval_seed=EVAL_SEED, device=DEVICE, batch_size=BATCH,
    )
    return view_consistency_diagnostics(z, DIAG_N_VIEWS)

In [ ]:
def train_one(lam, seed, synth_train, synth_val, tokenizer,
              epochs=EPOCHS, tag=VERSION_TAG):
    """Train one (lambda, seed) run. Training only — no symbolic eval."""
    seed_everything(seed)

    run_tag = f'lam{lam}_seed{seed}'
    run_dir = f'{CKPT_DIR}/{tag}/{run_tag}'
    CKPT_PATH = f'{run_dir}/latest.pt'
    BEST_PATH = f'{run_dir}/best.pt'

    if os.path.exists(CKPT_PATH):
        ck = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
        if ck['epoch'] >= epochs:
            print(f'{run_tag}: training complete (epoch {ck["epoch"]}), SKIPPING')
            return
    os.makedirs(run_dir, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'{run_tag} | n_views={N_VIEWS} | loss={SUBSAMPLE_LOSS} | '
          f'view_seed={TRAIN_VIEW_SEED} | tag={tag}')
    print(f'{"="*60}')

    model, encoder = build_model(tokenizer)

    g = torch.Generator()
    g.manual_seed(seed)
    # Exactly ONE loader here forks worker processes.  That is deliberate.
    #
    # A DataLoader worker inherits, at fork time, every object alive in the
    # parent -- including any other loader's live iterator.  When that worker
    # exits it runs the inherited iterator's __del__, and Process.is_alive()
    # asserts _parent_pid == os.getpid():
    #     AssertionError: can only test a child process
    # With a single forking loader there is no other iterator to inherit, so
    # the failure mode cannot arise.
    #
    # train: persistent_workers MUST stay False -- workers hold a forked copy
    #   of the dataset, so persistent ones would never observe
    #   `synth_train.epoch` and the views would silently freeze at epoch 0.
    # val:   num_workers=0.  Its clouds are deterministic and cached
    #   (cache=True in build_multiview_synthetic_splits), so after the first
    #   epoch this is just a dict lookup -- workers would buy nothing.
    train_loader = DataLoader(synth_train, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS,
                              persistent_workers=False,
                              pin_memory=True, worker_init_fn=seed_worker,
                              generator=g)
    val_loader = DataLoader(synth_val, batch_size=BATCH, shuffle=False,
                            num_workers=0)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    start_epoch = 1
    best_val = float('inf')
    best_val_acc = 0.0
    best_vc = {}
    history = {'train': [], 'val': []}

    if os.path.exists(CKPT_PATH):
        ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        start_epoch = ck['epoch'] + 1
        best_val = ck['best_val']
        best_val_acc = ck.get('best_val_acc', 0.0)
        best_vc = ck.get('best_view_consistency', {})
        history = ck['history']
        print(f'  Resuming from epoch {start_epoch} (best val: {best_val:.4f})')

    amp_ctx = amp_context()
    writer = SummaryWriter(log_dir=f'{LOG_DIR}/{tag}/{run_tag}')

    for epoch in range(start_epoch, epochs + 1):
        # Fresh, reproducible views for this epoch.
        synth_train.epoch = epoch

        model.train()
        train_loss_gen = 0
        train_loss_sub = 0
        train_sub_other = 0   # the mode we are NOT optimising, for reference
        pbar = tqdm(train_loader, desc=f'{run_tag} E{epoch}/{epochs}', leave=False)
        for batch in pbar:
            points_views = batch['points_views'].to(DEVICE, non_blocking=True)
            input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
            attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            with amp_ctx():
                # View 0 drives the ordinary symbolic-regression forward pass.
                out = model(points_views[:, 0], input_ids, attn_mask=attn_mask)
                loss_gen = out['loss']

                if lam > 0:
                    # z for view 0 is already computed; encode the rest.
                    z_views = [out['z_num']] + [
                        model.encoder(points_views[:, v])
                        for v in range(1, N_VIEWS)
                    ]
                    loss_sub = subsample_consistency_loss(
                        z_views, mode=SUBSAMPLE_LOSS)
                    loss = loss_gen + lam * loss_sub
                    # Track the other mode too — free, and shows how the
                    # two scales move relative to each other.
                    with torch.no_grad():
                        other = 'cosine' if SUBSAMPLE_LOSS == 'centered' else 'centered'
                        loss_sub_other = subsample_consistency_loss(
                            z_views, mode=other)
                else:
                    loss_sub = loss_sub_other = None
                    loss = loss_gen

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss_gen += loss_gen.item()
            postfix = {'ce': f'{loss_gen.item():.4f}'}
            global_step = (epoch - 1) * len(train_loader) + pbar.n
            writer.add_scalar('train/loss_step', loss_gen.item(), global_step)
            if loss_sub is not None:
                train_loss_sub += loss_sub.item()
                train_sub_other += loss_sub_other.item()
                postfix['sub'] = f'{loss_sub.item():.4f}'
                writer.add_scalar('train/subsample', loss_sub.item(), global_step)
                writer.add_scalar('train/subsample_other',
                                  loss_sub_other.item(), global_step)
                writer.add_scalar('train/subsample_weighted',
                                  (lam * loss_sub).item(), global_step)
            pbar.set_postfix(postfix)

        # tqdm.auto stores iter(train_loader), so `pbar` pins the train
        # workers alive.  Drop it before validation forks, or the val
        # workers inherit this iterator and fail in __del__ on exit.
        pbar.close()
        del pbar

        scheduler.step()
        train_avg = train_loss_gen / len(train_loader)
        history['train'].append(train_avg)
        history.setdefault('train_sub', []).append(train_loss_sub / len(train_loader))
        history.setdefault('train_sub_other', []).append(
            train_sub_other / len(train_loader))
        writer.add_scalar('train/loss_epoch', train_avg, epoch)
        writer.add_scalar('train/lr', scheduler.get_last_lr()[0], epoch)

        # ── Validation (token-weighted aggregation) ──
        if epoch % VAL_EVERY == 0 or epoch == epochs:
            model.eval()
            val_loss_sum = 0.0
            val_tokens_total = 0
            acc_correct = 0.0
            acc_total = 0.0
            with torch.no_grad(), amp_ctx():
                for batch in val_loader:
                    points    = batch['points'].to(DEVICE, non_blocking=True)
                    input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
                    attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)
                    out = model(points, input_ids, attn_mask=attn_mask)

                    n_tok = out['n_tokens']
                    val_loss_sum += out['loss'].item() * n_tok
                    val_tokens_total += n_tok

                    c, t = teacher_forced_counts(out['logits'], input_ids,
                                                 tokenizer.pad_id)
                    acc_correct += c
                    acc_total += t

            val_avg = val_loss_sum / max(val_tokens_total, 1)
            val_acc_avg = acc_correct / max(acc_total, 1)
            history['val'].append(val_avg)
            history.setdefault('val_acc', []).append(val_acc_avg)
            writer.add_scalar('val/loss', val_avg, epoch)
            writer.add_scalar('val/token_accuracy', val_acc_avg, epoch)

            # Representation consistency on fixed diagnostic views
            vc = view_consistency(model, synth_val)
            for k, v in vc.items():
                writer.add_scalar(f'val/{k}', v, epoch)
                history.setdefault(f'val_{k}', []).append(v)

            is_best = val_avg < best_val
            if is_best:
                best_val = val_avg
                best_val_acc = val_acc_avg
                best_vc = vc
            flag = ' * best' if is_best else ''
            print(f'  E{epoch}/{epochs} | train={train_avg:.4f} | '
                  f'val={val_avg:.4f}{flag} | acc={val_acc_avg*100:.1f}% | '
                  f'gap={vc["gap"]:.4f} gap_c={vc["gap_centered"]:.4f}')

            if is_best:
                torch.save({'model': model.state_dict(), 'epoch': epoch,
                            'val': val_avg, 'val_acc': val_acc_avg,
                            'view_consistency': vc}, BEST_PATH)

        torch.save({
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'epoch': epoch,
            'best_val': best_val,
            'best_val_acc': best_val_acc,
            'best_view_consistency': best_vc,
            'history': history,
            'lambda_subsample': lam,
            'subsample_loss': SUBSAMPLE_LOSS,
            'seed': seed,
            'n_views': N_VIEWS,
            'train_view_seed': TRAIN_VIEW_SEED,
        }, CKPT_PATH)

    writer.close()

    del train_loader, val_loader, model, encoder, optimizer, scheduler
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

In [ ]:
def eval_one(lam, seed, synth_val, synth_test, tokenizer, tag=VERSION_TAG):
    """Load the best checkpoint and evaluate on the full test set."""
    run_tag = f'lam{lam}_seed{seed}'
    run_dir = f'{CKPT_DIR}/{tag}/{run_tag}'
    metrics_path = f'{run_dir}/metrics.json'
    BEST_PATH = f'{run_dir}/best.pt'
    CKPT_PATH = f'{run_dir}/latest.pt'

    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            return json.load(f)

    model, encoder = build_model(tokenizer, dropout=0.0)
    ck_path = BEST_PATH if os.path.exists(BEST_PATH) else CKPT_PATH
    ck = torch.load(ck_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model'])
    model.eval()

    latest_ck = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
    best_val = ck.get('val', latest_ck.get('best_val', float('inf')))
    best_val_acc = ck.get('val_acc', latest_ck.get('best_val_acc', 0.0))
    vc = ck.get('view_consistency') or latest_ck.get('best_view_consistency') or {}
    if not vc:
        vc = view_consistency(model, synth_val)

    # Greedy decode over the FULL test set
    eval_loader = DataLoader(synth_test, batch_size=BATCH, shuffle=False)
    greedy_preds = []
    for batch in tqdm(eval_loader, desc=f'{run_tag} decode', leave=False):
        points = batch['points'].to(DEVICE)
        input_ids = batch['input_ids']
        preds = model.generate(points, tokenizer, max_new_tokens=MAX_SEQ)
        for j, pred_str in enumerate(preds):
            greedy_preds.append((tokenizer.decode(input_ids[j].tolist()), pred_str))

    res = evaluate_predictions(greedy_preds, synth_test, tokenizer)

    metrics = {
        'lambda': lam,
        'seed': seed,
        'run_tag': run_tag,
        'version_tag': tag,
        'n_views': N_VIEWS,
        'train_view_seed': TRAIN_VIEW_SEED,
        'subsample_loss': SUBSAMPLE_LOSS,
        'best_val_loss': best_val,
        'best_val_acc': best_val_acc,
        'same_fn_cos': vc.get('same_fn_cos', float('nan')),
        'diff_fn_cos': vc.get('diff_fn_cos', float('nan')),
        'cos_gap': vc.get('gap', float('nan')),
        'same_fn_cos_centered': vc.get('same_fn_cos_centered', float('nan')),
        'diff_fn_cos_centered': vc.get('diff_fn_cos_centered', float('nan')),
        'cos_gap_centered': vc.get('gap_centered', float('nan')),
        'greedy_exact_match': res['exact_match'],
        'greedy_token_acc': res['token_accuracy'],
        'greedy_algebraic_equiv': res['algebraic_equiv'],
        'greedy_r2_above_0.9': res['r2_above_0.9'],
        'mean_r2': res['mean_r2'],
        'median_r2': res['median_r2'],
        'n_parseable': res['n_parseable'],
        'n_total': res['n_total'],
        'details': res['details'],
    }
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)

    del model, encoder, eval_loader
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return metrics

## Sweep

Only run this once the smoke test looks right. 6 lambdas x 3 seeds = 18 runs.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Phase 1: train every run (no sympy in this phase)
# ══════════════════════════════════════════════════════════════
RUNS = [(l, s) for l in LAMBDA_VALUES for s in SEEDS]

print(f'Phase 1: training {len(RUNS)} runs...')
for lam, seed in RUNS:
    train_one(lam, seed, synth_train, synth_val, tokenizer)

# ══════════════════════════════════════════════════════════════
# Phase 2: evaluate every run on the full test set
# ══════════════════════════════════════════════════════════════
print(f'\n\n{"="*70}')
print(f'Phase 2: evaluating on the full test set ({len(synth_test)} eqs)...')
print(f'{"="*70}')

all_metrics = []
for i, (lam, seed) in enumerate(RUNS):
    run_tag = f'lam{lam}_seed{seed}'
    t0 = time.time()
    try:
        m = eval_one(lam, seed, synth_val, synth_test, tokenizer)
        print(f'  [{i+1}/{len(RUNS)}] {run_tag}: '
              f'exact={m["greedy_exact_match"]*100:.1f}% | '
              f'equiv={m["greedy_algebraic_equiv"]*100:.1f}% | '
              f'gap_c={m["cos_gap_centered"]:.3f} | {time.time()-t0:.0f}s')
        all_metrics.append(m)
    except Exception as e:
        print(f'  [{i+1}/{len(RUNS)}] {run_tag}: FAILED after '
              f'{time.time()-t0:.0f}s — {e}')
        all_metrics.append({
            'lambda': lam, 'seed': seed, 'run_tag': run_tag,
            'best_val_loss': float('nan'), 'best_val_acc': 0,
            'greedy_exact_match': 0, 'greedy_algebraic_equiv': 0,
            'greedy_r2_above_0.9': 0,
            'same_fn_cos': float('nan'), 'diff_fn_cos': float('nan'),
            'cos_gap': float('nan'), 'cos_gap_centered': float('nan'),
            'same_fn_cos_centered': float('nan'),
            'diff_fn_cos_centered': float('nan'),
        })
    gc.collect()

In [ ]:
# ── Per-run table ──
print(f'{"="*104}')
print(f'Subsample JEPA — {VERSION_TAG} | n_views={N_VIEWS} | n_test={len(synth_test)}')
print(f'{"="*104}')
print(f'\n{"lambda":>7} {"seed":>6} {"val_loss":>10} {"val_acc":>9} '
      f'{"exact":>8} {"equiv":>8} {"R2>.9":>8} {"same_c":>9} '
      f'{"diff_c":>9} {"gap_c":>9} {"gap_raw":>9}')
print('-' * 104)
for m in all_metrics:
    print(f'{m["lambda"]:>7.3f} {m["seed"]:>6} {m["best_val_loss"]:>10.4f} '
          f'{m.get("best_val_acc",0)*100:>8.1f}% '
          f'{m["greedy_exact_match"]*100:>7.1f}% '
          f'{m["greedy_algebraic_equiv"]*100:>7.1f}% '
          f'{m["greedy_r2_above_0.9"]*100:>7.1f}% '
          f'{m["same_fn_cos_centered"]:>9.4f} '
          f'{m["diff_fn_cos_centered"]:>9.4f} '
          f'{m["cos_gap_centered"]:>9.4f} {m["cos_gap"]:>9.4f}')

# ── Mean across seeds ──
print(f'\n{"lambda":>7} {"val_acc":>9} {"exact":>8} {"equiv":>8} {"R2>.9":>8} '
      f'{"same_c":>9} {"diff_c":>9} {"gap_c":>9} {"gap_raw":>9} {"n":>4}')
print('-' * 88)
mean = lambda rs, k: float(np.mean([r[k] for r in rs]))
for lam in LAMBDA_VALUES:
    rs = [m for m in all_metrics if m['lambda'] == lam]
    if not rs:
        continue
    print(f'{lam:>7.3f} {mean(rs,"best_val_acc")*100:>8.1f}% '
          f'{mean(rs,"greedy_exact_match")*100:>7.1f}% '
          f'{mean(rs,"greedy_algebraic_equiv")*100:>7.1f}% '
          f'{mean(rs,"greedy_r2_above_0.9")*100:>7.1f}% '
          f'{mean(rs,"same_fn_cos_centered"):>9.4f} '
          f'{mean(rs,"diff_fn_cos_centered"):>9.4f} '
          f'{mean(rs,"cos_gap_centered"):>9.4f} '
          f'{mean(rs,"cos_gap"):>9.4f} {len(rs):>4}')

## Plots

In [ ]:
import matplotlib.pyplot as plt

# Representation consistency and recovery vs lambda (mean +- range over seeds)
def _agg(key):
    mu, lo, hi = [], [], []
    for lam in LAMBDA_VALUES:
        vals = [m[key] for m in all_metrics if m['lambda'] == lam]
        vals = [v for v in vals if v is not None and np.isfinite(v)]
        if not vals:
            vals = [np.nan]
        mu.append(np.mean(vals)); lo.append(np.min(vals)); hi.append(np.max(vals))
    mu, lo, hi = map(np.array, (mu, lo, hi))
    return mu, mu - lo, hi - mu

x = np.arange(len(LAMBDA_VALUES))
labels = [str(l) for l in LAMBDA_VALUES]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for key, lab in [('same_fn_cos_centered', 'same-function'),
                 ('diff_fn_cos_centered', 'different-function')]:
    mu, el, eh = _agg(key)
    axes[0].errorbar(x, mu, yerr=[el, eh], marker='o', capsize=3, label=lab)
axes[0].set_title('Encoder cosine (mean-centered)'); axes[0].legend(fontsize=8)

for key, lab, c in [('cos_gap_centered', 'centered', 'tab:green'),
                    ('cos_gap', 'raw', 'tab:gray')]:
    mu, el, eh = _agg(key)
    axes[1].errorbar(x, mu, yerr=[el, eh], marker='o', capsize=3,
                     color=c, label=lab)
axes[1].set_title('same - different (headline metric)'); axes[1].legend(fontsize=8)

for key, lab in [('greedy_algebraic_equiv', 'equiv'),
                 ('greedy_exact_match', 'exact'),
                 ('greedy_r2_above_0.9', 'R2>0.9')]:
    mu, el, eh = _agg(key)
    axes[2].errorbar(x, mu * 100, yerr=[el * 100, eh * 100],
                     marker='o', capsize=3, label=lab)
axes[2].set_title('Symbolic recovery (%)'); axes[2].legend(fontsize=8)

for ax in axes:
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_xlabel('lambda'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Training curves per run
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for lam in LAMBDA_VALUES:
    for seed in SEEDS:
        p = f'{CKPT_DIR}/{VERSION_TAG}/lam{lam}_seed{seed}/latest.pt'
        if not os.path.exists(p):
            continue
        h = torch.load(p, map_location='cpu', weights_only=False)['history']
        label = f'l={lam} s={seed}'
        a = 0.5 if len(SEEDS) > 1 else 1.0
        axes[0].plot(h['train'], label=label, alpha=a)
        axes[1].plot(h['val'], label=label, alpha=a)
        if h.get('val_acc'):
            axes[2].plot([v * 100 for v in h['val_acc']], label=label, alpha=a)
        if h.get('val_gap_centered'):
            axes[3].plot(h['val_gap_centered'], label=label, alpha=a)

for ax, t in zip(axes, ['Train CE', 'Val CE', 'Val token acc %',
                        'same - diff cosine (centered)']):
    ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
    ax.legend(fontsize=6)
plt.tight_layout(); plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## Inspect predictions

In [ ]:
def to_infix(prefix_str):
    """Prefix string -> infix via SymPy. Returns (infix, error)."""
    try:
        expr, _ = prefix_to_sympy(prefix_str)
        return str(expr), None
    except Exception as e:
        return None, str(e)

def inspect_run(run_tag, n_show=10, tag=VERSION_TAG):
    path = f'{CKPT_DIR}/{tag}/{run_tag}/metrics.json'
    if not os.path.exists(path):
        print(f'{run_tag}: no metrics.json'); return
    with open(path) as f:
        m = json.load(f)
    details = m.get('details', [])
    print(f'\n{"="*80}')
    print(f'{run_tag} | exact={m["greedy_exact_match"]*100:.1f}% | '
          f'equiv={m["greedy_algebraic_equiv"]*100:.1f}% | '
          f'gap_c={m["cos_gap_centered"]:.3f}')
    print(f'{"="*80}')
    for i, d in enumerate(details[:n_show]):
        gt, _ = to_infix(d['gt'])
        pred, err = to_infix(d['pred'])
        r2 = f'{d["r2"]:.4f}' if d.get('r2') is not None else 'N/A'
        status = ('EXACT' if d['exact'] else 'EQUIV' if d.get('equiv')
                  else 'PARSEABLE' if d.get('parseable') else 'UNPARSEABLE')
        print(f'\n-- [{i}] {status} | R2={r2}')
        print(f'  GT:   {gt}')
        print(f'  Pred: {pred if err is None else "PARSE FAILED: " + err}')

for lam in LAMBDA_VALUES:
    inspect_run(f'lam{lam}_seed{SEEDS[0]}', n_show=5)